# Unit 1: PyTorch 基础

## 学习目标
- 理解张量 (Tensor) 的概念和基本操作
- 掌握自动求导 (Autograd) 机制
- 学会在 GPU 上运行计算
- 实现一个简单的梯度下降示例

## 1.1 什么是张量 (Tensor)

张量是 PyTorch 中最基本的数据结构，可以理解为**多维数组**。
- 标量 (Scalar): 0 维张量
- 向量 (Vector): 1 维张量
- 矩阵 (Matrix): 2 维张量
- 高阶张量: 3 维及以上

PyTorch 的张量与 NumPy 的 ndarray 非常相似，但额外支持：
- **GPU 加速计算**
- **自动微分** (Autograd)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1.2 创建张量

In [ ]:
scalar = torch.tensor(3.14)
print(f"标量: {scalar}, shape: {scalar.shape}, dim: {scalar.dim()}")

vector = torch.tensor([1.0, 2.0, 3.0])
print(f"向量: {vector}, shape: {vector.shape}")

matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(f"矩阵:\n{matrix}, shape: {matrix.shape}")

tensor_3d = torch.randn(2, 3, 4)
print(f"3D 张量 shape: {tensor_3d.shape}")

In [ ]:
zeros = torch.zeros(3, 4)
ones = torch.ones(3, 4)
rand = torch.rand(3, 4)
randn = torch.randn(3, 4)
arange = torch.arange(0, 10, 2)
linspace = torch.linspace(0, 1, 5)

print(f"zeros:\n{zeros}\n")
print(f"rand (uniform [0,1)):\n{rand}\n")
print(f"randn (standard normal):\n{randn}\n")
print(f"arange: {arange}")
print(f"linspace: {linspace}")

## 1.3 张量属性与数据类型

In [ ]:
t = torch.randn(2, 3, 4)
print(f"shape: {t.shape}")
print(f"dtype: {t.dtype}")
print(f"device: {t.device}")
print(f"requires_grad: {t.requires_grad}")
print(f"numel (元素总数): {t.numel()}")

int_tensor = torch.tensor([1, 2, 3], dtype=torch.int64)
float_tensor = torch.tensor([1, 2, 3], dtype=torch.float32)
print(f"\nint64: {int_tensor.dtype}, float32: {float_tensor.dtype}")

t2 = t.float()
t3 = t.to(torch.float64)
print(f"类型转换: {t.dtype} -> {t3.dtype}")

## 1.4 张量运算

张量支持丰富的数学运算，与 NumPy 语法高度一致。

In [ ]:
a = torch.randn(2, 3)
b = torch.randn(2, 3)

print(f"a + b:\n{a + b}\n")
print(f"a * b (逐元素):\n{a * b}\n")
print(f"a @ b.T (矩阵乘法):\n{a @ b.T}\n")
print(f"torch.matmul(a, b.T):\n{torch.matmul(a, b.T)}")

In [ ]:
x = torch.randn(3, 4)
print(f"原始: shape={x.shape}\n{x}")
print(f"\nmean: {x.mean():.4f}")
print(f"std: {x.std():.4f}")
print(f"sum: {x.sum():.4f}")
print(f"max: {x.max():.4f}")

print(f"\n按列求 mean (dim=0): {x.mean(dim=0)}")
print(f"按行求 mean (dim=1): {x.mean(dim=1)}")

In [ ]:
x = torch.randn(2, 3, 4)
print(f"原始 shape: {x.shape}")

print(f"reshape(4, 6): {x.reshape(4, 6).shape}")
print(f"reshape(4, -1): {x.reshape(4, -1).shape}")
print(f"view(2, 12): {x.view(2, 12).shape}")
print(f"permute(2, 0, 1): {x.permute(2, 0, 1).shape}")
print(f"unsqueeze(0): {x.unsqueeze(0).shape}")
print(f"squeeze on dim=0: {x.unsqueeze(0).squeeze(0).shape}")

### 索引与切片

In [ ]:
x = torch.arange(12).reshape(3, 4)
print(f"原始:\n{x}")
print(f"\n第一行: {x[0]}")
print(f"前两行、后两列:\n{x[:2, -2:]}")
print(f"\n布尔索引 (x > 5): {x[x > 5]}")

## 1.5 NumPy 互操作

In [ ]:
np_arr = np.array([[1, 2], [3, 4]], dtype=np.float32)
tensor = torch.from_numpy(np_arr)
print(f"NumPy -> Tensor: {tensor}")

back_to_np = tensor.numpy()
print(f"Tensor -> NumPy: {back_to_np}")

print(f"共享内存? {np_arr[0, 0]}")
tensor[0, 0] = 99
print(f"修改 tensor 后 NumPy: {np_arr[0, 0]}")

## 1.6 GPU 加速

PyTorch 最大的优势之一是可以在 GPU 上无缝运行计算。

In [ ]:
if torch.cuda.is_available():
    x_cpu = torch.randn(1000, 1000)
    x_gpu = x_cpu.to("cuda")
    print(f"CPU tensor device: {x_cpu.device}")
    print(f"GPU tensor device: {x_gpu.device}")

    import time
    t0 = time.time()
    for _ in range(100):
        _ = x_cpu @ x_cpu
    cpu_time = time.time() - t0

    t0 = time.time()
    for _ in range(100):
        _ = x_gpu @ x_gpu
    torch.cuda.synchronize()
    gpu_time = time.time() - t0

    print(f"\nCPU 矩阵乘法 100 次: {cpu_time:.3f}s")
    print(f"GPU 矩阵乘法 100 次: {gpu_time:.3f}s")
    print(f"加速比: {cpu_time / gpu_time:.1f}x")
else:
    print("CUDA 不可用，跳过 GPU 对比")

## 1.7 自动求导 (Autograd)

Autograd 是 PyTorch 的核心机制之一，它能**自动计算梯度**，是训练神经网络的基石。

工作流程：
1. 设置 `requires_grad=True` 标记需要追踪的张量
2. 执行前向计算，PyTorch 自动构建**计算图**
3. 调用 `.backward()` 自动计算梯度
4. 梯度存储在 `.grad` 属性中

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1

print(f"y = x^2 + 3x + 1")
print(f"当 x = {x.item():.1f} 时, y = {y.item():.1f}")

y.backward()
print(f"dy/dx = 2x + 3 = {x.grad.item():.1f}")
print(f"手动计算: 2*{x.item():.1f} + 3 = {2*x.item()+3:.1f}")

In [ ]:
x = torch.randn(3, requires_grad=True)
w = torch.tensor([2.0, 3.0, 1.0], requires_grad=True)

y = (w * x).sum()
print(f"y = sum(w * x) = {y.item():.4f}")
y.backward()

print(f"dy/dx = w = {x.grad}")
print(f"dy/dw = x = {w.grad}")

In [ ]:
x = torch.randn(2, 2, requires_grad=True)
print(f"x:\n{x}")

y = x ** 2
z = y.mean()

print(f"\ny = x^2")
print(f"y:\n{y}")
print(f"\nz = mean(y) = {z:.4f}")

z.backward()
print(f"\ndz/dx:\n{x.grad}")
print(f"手动验证: mean(x^2) 对 x 求导 = 2x/4 = x/2")
print(f"x/2:\n{x / 2}")

### 梯度清零与 no_grad

In [ ]:
x = torch.tensor([3.0], requires_grad=True)
for i in range(3):
    y = x ** 2
    y.backward()
    print(f"Step {i}: grad = {x.grad.item():.1f} (累积!)")

print("\n使用 grad.zero_() 清零:")
x = torch.tensor([3.0], requires_grad=True)
for i in range(3):
    y = x ** 2
    y.backward()
    print(f"Step {i}: grad before zero = {x.grad.item():.1f}", end="")
    x.grad.zero_()
    print(f", after zero = {x.grad.item():.1f}")

print("\ntorch.no_grad() 用于推理/评估，不构建计算图:")
with torch.no_grad():
    y = x ** 2
    print(f"在 no_grad 中: y = {y.item():.1f}, requires_grad = {y.requires_grad}")

## 1.8 实战：用梯度下降求函数最小值

目标：找到 $f(x) = (x-3)^2 + 2$ 的最小值。

解析解：当 $x=3$ 时，最小值为 $2$。

我们用梯度下降来逼近这个解。

In [ ]:
x = torch.tensor([0.0], requires_grad=True)
lr = 0.1
history = []

for step in range(50):
    y = (x - 3) ** 2 + 2

    if x.grad is not None:
        x.grad.zero_()
    y.backward()

    history.append((step, x.item(), y.item(), x.grad.item()))

    with torch.no_grad():
        x -= lr * x.grad

for s, xi, yi, gi in history[:5]:
    print(f"Step {s:2d}: x={xi:.4f}, f(x)={yi:.4f}, grad={gi:.4f}")
print("...")
for s, xi, yi, gi in history[-3:]:
    print(f"Step {s:2d}: x={xi:.4f}, f(x)={yi:.4f}, grad={gi:.4f}")

In [ ]:
steps = [h[0] for h in history]
x_vals = [h[1] for h in history]
y_vals = [h[2] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

xs = np.linspace(-1, 7, 100)
ax1.plot(xs, (xs - 3) ** 2 + 2, "b-", label="f(x)")
ax1.scatter(x_vals, y_vals, c=range(len(x_vals)), cmap="viridis", s=10)
ax1.set_xlabel("x")
ax1.set_ylabel("f(x)")
ax1.set_title("Gradient Descent Path")
ax1.legend()

ax2.plot(steps, y_vals, "r-")
ax2.axhline(y=2, color="gray", linestyle="--", label="minimum = 2")
ax2.set_xlabel("Step")
ax2.set_ylabel("f(x)")
ax2.set_title("Convergence")
ax2.legend()

plt.tight_layout()
plt.show()

## 1.9 单元小结

| 概念 | 要点 |
|------|------|
| **Tensor** | 多维数组，支持 GPU 加速和自动求导 |
| **创建张量** | `torch.tensor()`, `torch.zeros()`, `torch.randn()`, `torch.arange()` |
| **运算** | `+`, `*`, `@`, `mean()`, `sum()`, `reshape()`, `permute()` |
| **GPU** | `.to('cuda')` 将张量移到 GPU |
| **Autograd** | `requires_grad=True` + `.backward()` 自动计算梯度 |
| **梯度下降** | `x -= lr * x.grad` 更新参数 |

### 思考题
1. `view()` 和 `reshape()` 有什么区别？什么时候必须用 `reshape()`？
2. 如果不调用 `grad.zero_()` 会发生什么？
3. `.detach()` 的作用是什么？它与 `torch.no_grad()` 有什么不同？